In [2]:
import functools

import ee
import geemap

In [3]:
ee.Initialize()

In [5]:
def rasterize_poly(feature: ee.Feature, id_field: str) -> ee.Image:
    fraction = ee.Image.constant(1).clip(feature.geometry()).rename("fraction")

    poly_id_val = ee.Number(feature.get(id_field))
    poly_id = ee.Image.constant(poly_id_val).rename("poly_id")

    return ee.Image.cat([fraction, poly_id])


def get_overlap_with_ids(polygons: ee.FeatureCollection, id_field: str):
    map_func = functools.partial(rasterize_poly, id_field=id_field)
    poly_images = ee.ImageCollection(polygons.map(map_func))

    array_image = poly_images.toArray()

    return array_image

In [ ]:
col = ee.ImageCollection("JRC/GHSL/P2023A/GHS_POP")
img = col.filterDate("2020-01-01", "2020-12-31").first()

In [ ]:
chunked_df = [df_agebs.iloc[i : i + 1000] for i in range(0, len(df_agebs), 1000)]
features = [geemap.geopandas_to_ee(gdf) for gdf in chunked_df]


results = []
for feature in features:
    computed = ee.data.computeFeatures(
        {
            "expression": img.reduceRegions(
                collection=feature,
                reducer=ee.Reducer.sum(),
                crs=img.projection(),
            ),
            "fileFormat": "PANDAS_DATAFRAME",
        },
    )
    results.append(computed)

EEException: Request payload size exceeds the limit: 10485760 bytes.